# Fatigue Load Measures Analysis

**Exploring rolling, decaying, and cumulative fatigue load measures to better predict defensive-quality signals.**

This notebook extends the main analysis by computing four time-aware fatigue load measures:
1. **Rolling 10-minute window** — average load across ~2 adjacent blocks (centred)
2. **Decaying 15-minute window** — exponential weighted moving average (half-life ~7.5 min)
3. **Cumulative per half** — running sum of load within each match half (Phase 1 / Phase 2)
4. **Cumulative per match** — running sum across the entire match

These are compared against the raw block-level load to determine which measure best predicts five defensive-quality signals: positional_drift, shift_latency, pressing_accuracy, transition_latency, and reorientation_rate.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams["figure.dpi"] = 150
matplotlib.rcParams["figure.figsize"] = (12, 6)
matplotlib.rcParams["font.size"] = 11

OUT_DIR = Path("outputs/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("All imports loaded.")

In [ ]:
# Load the real unified fatigue dataset
import os
candidates = [
    "unified_fatigue_dataset.parquet",
    "../unified_fatigue_dataset.parquet",
    os.path.expanduser("~/.openclaw/workspace/speed-check/unified_fatigue_dataset.parquet"),
    os.path.expanduser("~/.openclaw/workspace/speed-check/focus-fatigue/unified_fatigue_dataset.parquet"),
]
df = None
for p in candidates:
    try:
        df = pd.read_parquet(p)
        print(f"Loaded from: {p}")
        break
    except (FileNotFoundError, OSError):
        continue
if df is None:
    raise FileNotFoundError("Could not find unified_fatigue_dataset.parquet")

print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(f"Games: {df['game_id'].nunique()}, Players: {df['player_id'].nunique()}, Blocks: {df['block_id'].nunique()}")

## 1. Compute Fatigue Load Measures

We compute four time-aware fatigue load measures using `pressure_composite` as the primary load variable.

**Key data-structure decisions:**
- Block numbers reset per phase (phase 1 and phase 2 each start at block 0)
- For rolling/decaying measures we sort by phase and block_num within each (game, player) group
- Cumulative-per-half resets at phase boundary; cumulative-per-match does not
- Rolling/decaying windows are computed within each phase (the half-time break naturally resets fatigue context)

In [ ]:
# --- Compute all fatigue load measures ---

# Half-life and alpha for EWMA
HALF_LIFE_MIN = 7.5
BLOCK_DURATION_MIN = 5.0
ALPHA = 1 - np.exp(-np.log(2) / (HALF_LIFE_MIN / BLOCK_DURATION_MIN))

def _phase_loads(phase_subset):
    """Compute rolling, EWMA, and cumulative within a single phase."""
    ps = phase_subset.copy()
    n = len(ps)
    load = ps["pressure_composite"].values
    
    # Rolling 10-min (centred, +/-1 block)
    rolling = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - 1)
        hi = min(n, i + 2)
        rolling[i] = np.mean(load[lo:hi])
    ps["load_rolling_10min"] = rolling
    
    # EWMA (resets at start of each phase)
    ewma = np.full(n, np.nan)
    if n > 0:
        ewma[0] = load[0]
        for i in range(1, n):
            ewma[i] = ALPHA * load[i] + (1 - ALPHA) * ewma[i-1]
    ps["load_decaying_15min"] = ewma
    
    # Cumulative within this phase
    ps["load_cumulative_half"] = np.cumsum(load)
    
    return ps

def compute_fatigue_loads(group):
    """Compute all fatigue load measures for a single (game,player) group."""
    g = group.sort_values(["phase", "block_num"]).copy()
    n = len(g)
    
    # Rolling, EWMA, and per-half cumulative: compute within each phase
    # (half-time naturally resets fatigue context)
    phase_chunks = []
    for phase_val in [1, 2]:
        mask = g["phase"] == phase_val
        if mask.any():
            phase_chunks.append(_phase_loads(g[mask]))
    
    if len(phase_chunks) > 0:
        result = pd.concat(phase_chunks).sort_index()
        # Transfer computed columns back to g
        for col in ["load_rolling_10min", "load_decaying_15min", "load_cumulative_half"]:
            g[col] = result[col].values
    else:
        for col in ["load_rolling_10min", "load_decaying_15min", "load_cumulative_half"]:
            g[col] = np.nan
    
    # Cumulative per match (no reset across phases)
    g["load_cumulative_match"] = g["pressure_composite"].values
    cumsum = 0.0
    for i in range(n):
        cumsum += g.iloc[i]["pressure_composite"]
        g.iloc[i, g.columns.get_loc("load_cumulative_match")] = cumsum
    
    return g

print("Computing fatigue load measures...")
df_with_loads = (df
    .groupby(["game_id", "player_id"], group_keys=True)
    .apply(compute_fatigue_loads)
    .reset_index()
)

print("Computing fatigue load measures...")
df_with_loads = (df
    .groupby(["game_id", "player_id"], group_keys=True)
    .apply(compute_fatigue_loads)
    .reset_index()
)

print(f"\nMeasures computed. Shape: {df_with_loads.shape}")

cols_show = ["game_id","player_id","phase","block_num","pressure_composite",
             "load_rolling_10min","load_decaying_15min","load_cumulative_half","load_cumulative_match"]
print(f"\nSample rows (first 8):")
print(df_with_loads[cols_show].head(8).to_string())

measure_cols = ["pressure_composite", "load_rolling_10min", "load_decaying_15min",
                "load_cumulative_half", "load_cumulative_match"]
print(f"\nMeasure statistics:")
print(df_with_loads[measure_cols].describe().round(4))

## 2. Compare Predictive Power Against Defensive-Quality Signals

We evaluate how well each fatigue load measure predicts five defensive-quality signals using linear regression (R-squared).

**Measures:**
- `pressure_composite` (raw block load) — baseline
- `load_rolling_10min` — centred 3-block average
- `load_decaying_15min` — EWMA with 7.5-min half-life
- `load_cumulative_half` — running sum within each half
- `load_cumulative_match` — running sum across entire match

**Target signals:** positional_drift, shift_latency, pressing_accuracy, transition_latency, reorientation_rate

In [ ]:
# --- Prepare data for comparison ---
target_signals = [
    "positional_drift",
    "shift_latency",
    "pressing_accuracy",
    "transition_latency",
    "reorientation_rate"
]

measure_cols = [
    "pressure_composite",
    "load_rolling_10min",
    "load_decaying_15min",
    "load_cumulative_half",
    "load_cumulative_match",
]

measure_labels = [
    "Block Load (raw)",
    "Rolling 10-min",
    "Decaying 15-min",
    "Cumulative per Half",
    "Cumulative per Match",
]

results_rows = []

for signal in target_signals:
    valid = df_with_loads[["game_id", "player_id", signal] + measure_cols].dropna(subset=[signal])

    for measure, label in zip(measure_cols, measure_labels):
        m_valid = valid.dropna(subset=[measure])
        if len(m_valid) < 10:
            results_rows.append({
                "target_signal": signal,
                "measure": label,
                "measure_code": measure,
                "r_squared": np.nan,
                "pearson_r": np.nan,
                "n_obs": len(m_valid),
                "slope": np.nan,
                "intercept": np.nan,
            })
            continue

        x = m_valid[measure].values
        y = m_valid[signal].values

        slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
        r_sq = r_val ** 2

        results_rows.append({
            "target_signal": signal,
            "measure": label,
            "measure_code": measure,
            "r_squared": r_sq,
            "pearson_r": r_val,
            "p_value": p_val,
            "n_obs": len(m_valid),
            "slope": slope,
            "intercept": intercept,
        })

results_df = pd.DataFrame(results_rows)

print("=" * 60)
print("PREDICTIVE POWER COMPARISON (R-squared)")
print("=" * 60)
print()

r2_pivot = results_df.pivot_table(
    index="measure", columns="target_signal", values="r_squared", aggfunc="first"
)
r2_pivot = r2_pivot[target_signals]
r2_pivot = r2_pivot.reindex(measure_labels)

print(r2_pivot.round(6).to_string(float_format=lambda x: f"{x:.6f}" if not pd.isna(x) else "   N/A   "))
print()

print("Best fatigue measure for each signal:")
best_rows = []
for signal in target_signals:
    sr = results_df[results_df["target_signal"] == signal].dropna(subset=["r_squared"])
    if len(sr) > 0:
        best = sr.loc[sr["r_squared"].idxmax()]
        best_rows.append(best)
        print(f"  {signal:<25s} -> {best['measure']:<25s}  R2 = {best['r_squared']:.6f}  p = {best['p_value']:.6f}")

best_df = pd.DataFrame(best_rows)

avg_r2 = results_df.groupby("measure")["r_squared"].mean().sort_values(ascending=False)
print(f"\nOverall average R-squared per measure:")
for m, r2 in avg_r2.items():
    print(f"  {m:<30s} {r2:.6f}")
print(f"\nBest overall measure: {avg_r2.index[0]} (mean R2 = {avg_r2.iloc[0]:.6f})")

results_df.to_csv(OUT_DIR / "fatigue_load_predictive_power.csv", index=False)
print(f"\nResults saved to: {OUT_DIR / 'fatigue_load_predictive_power.csv'}")

## 3. Visualise Predictive Power Comparison

In [ ]:
# --- Grouped bar chart: R-squared comparison ---
fig, ax = plt.subplots(figsize=(14, 7))

plot_data = results_df.pivot_table(
    index="measure", columns="target_signal", values="r_squared", aggfunc="first"
)
plot_data = plot_data.reindex(measure_labels)
plot_data = plot_data[target_signals]

x = np.arange(len(measure_labels))
n_signals = len(target_signals)
width = 0.8 / n_signals
colors = plt.cm.Set2(np.linspace(0, 1, n_signals))

for i, signal in enumerate(target_signals):
    offset = (i - n_signals/2 + 0.5) * width
    values = plot_data[signal].values
    bars = ax.bar(x + offset, values, width, label=signal, color=colors[i], alpha=0.85)
    for j, (v, bar) in enumerate(zip(values, bars)):
        if not pd.isna(v) and v > 0.001:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                    f"{v:.4f}", ha="center", va="bottom", fontsize=7, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels(measure_labels, rotation=15, ha="right")
ax.set_ylabel("R-squared")
ax.set_title("Predictive Power: Which Fatigue Measure Best Explains Each Signal?", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(OUT_DIR / "fatigue_load_r2_comparison.png", bbox_inches="tight", dpi=150)
plt.close()
print("Saved: fatigue_load_r2_comparison.png")

In [ ]:
# --- Best measure per signal (horizontal bar chart) ---
fig, ax = plt.subplots(figsize=(12, 6))

best_per_signal = []
for signal in target_signals:
    sig_data = results_df[results_df["target_signal"] == signal].dropna(subset=["r_squared"])
    if len(sig_data) > 0:
        best_idx = sig_data["r_squared"].idxmax()
        best_per_signal.append(sig_data.loc[best_idx])

best_per_signal = pd.DataFrame(best_per_signal)

measure_colors_map = {label: c for label, c in zip(measure_labels,
                      plt.cm.tab10(np.linspace(0, 1, len(measure_labels))))}

y_pos = np.arange(len(target_signals))
bars = ax.barh(y_pos, best_per_signal["r_squared"].values,
               color=[measure_colors_map[m] for m in best_per_signal["measure"].values],
               height=0.6)

for i, (_, row) in enumerate(best_per_signal.iterrows()):
    ax.text(row["r_squared"] + 0.0003, i,
            f"{row['measure']}  (R2={row['r_squared']:.5f})",
            va="center", fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(best_per_signal["target_signal"].values)
ax.set_xlabel("R-squared")
ax.set_title("Best Fatigue Measure for Each Defensive Signal", fontweight="bold")
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=color, label=label) for label, color in measure_colors_map.items()]
ax.legend(handles=legend_elems, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fatigue_load_best_measure_per_signal.png", bbox_inches="tight", dpi=150)
plt.close()
print("Saved: fatigue_load_best_measure_per_signal.png")

## 4. Fatigue Load Trajectory Visualisation

Sample trajectories for high-block-count players showing how the different load measures evolve across a match.

In [ ]:
# --- Sample player load trajectories ---
player_block_counts = df_with_loads.groupby("player_id").size().sort_values(ascending=False)
top_players = player_block_counts.head(6).index.tolist()

print(f"Sample players: {top_players}")

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, pid in enumerate(top_players):
    ax = axes[i]
    pdata = df_with_loads[df_with_loads["player_id"] == pid].sort_values(["phase", "block_num"]).copy()

    ax.plot(pdata.index, pdata["pressure_composite"], "o-", label="Block Load", alpha=0.7, linewidth=1.5, markersize=4)
    ax.plot(pdata.index, pdata["load_rolling_10min"], "s--", label="Rolling 10-min", alpha=0.7, linewidth=1.5)
    ax.plot(pdata.index, pdata["load_decaying_15min"], "^-.", label="Decaying 15-min", alpha=0.7, linewidth=1.5)

    ax2 = ax.twinx()
    ax2.plot(pdata.index, pdata["load_cumulative_half"], "x-", label="Cumul/Half", alpha=0.5, linewidth=1, color="purple")
    ax2.plot(pdata.index, pdata["load_cumulative_match"], "*-", label="Cumul/Match", alpha=0.5, linewidth=1, color="brown")
    ax2.set_ylabel("Cumulative Load", fontsize=9)
    ax2.tick_params(labelsize=8)

    # Phase boundary
    if (pdata["phase"] == 2).any():
        p1_max_idx = pdata[pdata["phase"] == 1].index[-1] if (pdata["phase"] == 1).any() else None
        if p1_max_idx is not None:
            ax.axvline(x=p1_max_idx + 0.5, color="gray", linestyle=":", alpha=0.5)
            ax.text(p1_max_idx + 0.5, ax.get_ylim()[1] * 0.95, "HT", ha="center", fontsize=8, alpha=0.6)

    ax.set_title(f"Player {pid}", fontweight="bold", fontsize=10)
    ax.set_xlabel("Block Sequence")
    ax.set_ylabel("Pressure / Smoothed Load")
    ax.tick_params(labelsize=8)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR / "fatigue_load_trajectories.png", bbox_inches="tight", dpi=150)
plt.close()
print("Saved: fatigue_load_trajectories.png")

## 5. Measure Correlation Matrix

How do the different fatigue load measures correlate with each other? This helps understand redundancy vs. unique information.

In [ ]:
# --- Correlation matrix ---
corr_cols = ["pressure_composite", "load_rolling_10min", "load_decaying_15min",
             "load_cumulative_half", "load_cumulative_match"]
corr = df_with_loads[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".3f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, ax=ax,
            xticklabels=measure_labels, yticklabels=measure_labels)
ax.set_title("Fatigue Load Measure Correlations", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "fatigue_load_correlation_heatmap.png", bbox_inches="tight", dpi=150)
plt.close()
print("Saved: fatigue_load_correlation_heatmap.png")

print("\nCorrelation matrix:")
print(corr.round(4).to_string())

## 6. Physical Load Analysis

We analyse the **physical_load** signal (total distance, HSR, sprint metrics) as a potential confounding control variable. Physical exertion may independently degrade defensive quality, so we need to check:

1. How does physical_load correlate with the fatigue load measures?
2. How does physical_load correlate with the cognitive target signals?
3. Does controlling for physical_load change the fatigue→signal relationship?

**Columns added by the physical_load signal:** total_distance, hsr_distance, sprint_distance, hsr_efforts, avg_speed, max_speed.

In [ ]:
# --- Check for physical_load data ---
physical_cols = [
    "total_distance", "hsr_distance", "sprint_distance",
    "hsr_efforts", "avg_speed", "max_speed",
]
existing_physical = [c for c in physical_cols if c in df_with_loads.columns]
print(f"Physical load columns in dataset: {existing_physical}")
print(f"Missing: {[c for c in physical_cols if c not in df_with_loads.columns]}")

if existing_physical:
    print(f"\nPhysical load stats:")
    print(df_with_loads[existing_physical].describe().round(3))
else:
    print("\n⚠️  Physical load data not yet in unified dataset.")
    print("To add it:")
    print("  1. Run pipeline with physical_load signal (auto-detected via @register_signal)")
    print("  2. Re-run src/merge_outputs.py to merge physical_load into the unified dataset")
    print("  3. Re-run this notebook")


In [ ]:
# --- Physical load as a control variable in cognitive fatigue models ---
if existing_physical:
    print("=" * 70)
    print("PHYSICAL LOAD AS CONTROL VARIABLE")
    print("=" * 70)

    # For each target signal, run two models:
    #   Model A: signal ~ pressure_composite
    #   Model B: signal ~ pressure_composite + total_distance
    # Compare R\u00b2 to see how much physical load confounds the relationship

    import statsmodels.api as sm

    control_results = []
    targets = [
        "positional_drift", "shift_latency", "pressing_accuracy",
        "transition_latency", "reorientation_rate",
    ]

    for signal in targets:
        valid = df_with_loads[[signal, "pressure_composite"] + existing_physical].dropna()
        if len(valid) < 20:
            continue

        y = valid[signal].values

        # Model A: pressure_composite alone
        X_a = sm.add_constant(valid["pressure_composite"].values)
        model_a = sm.OLS(y, X_a).fit()

        # Model B: pressure_composite + physical_load (total_distance)
        X_b = sm.add_constant(valid[["pressure_composite", "total_distance"]].values)
        model_b = sm.OLS(y, X_b).fit()

        # Model C: physical_load alone
        X_c = sm.add_constant(valid["total_distance"].values)
        model_c = sm.OLS(y, X_c).fit()

        control_results.append({
            "target_signal": signal,
            "n_obs": len(valid),
            "R2_pressure_only": model_a.rsquared,
            "R2_physical_only": model_c.rsquared,
            "R2_pressure_plus_physical": model_b.rsquared,
            "R2_improvement": model_b.rsquared - model_a.rsquared,
            "p_pressure": model_b.pvalues[1],
            "p_physical": model_b.pvalues[2] if len(model_b.pvalues) > 2 else None,
        })

    control_df = pd.DataFrame(control_results)
    print()
    print(f"{\"Target Signal\":<25} {\"R\u00b2 Pressure\":>12} {\"+ Physical\":>12} {\"\u0394R\u00b2\":>8} {\"p(physical)\" :>12}")
    print(f"{'─' * 25} {'─' * 12} {'─' * 12} {'─' * 8} {'─' * 12}")
    for _, row in control_df.iterrows():
        p_str = f"{row["p_physical"]:.4f}" if row["p_physical"] is not None else "N/A"
        print(f"{row["target_signal"]:<25} {row["R2_pressure_only"]:>12.6f} "
              f"{row["R2_pressure_plus_physical"]:>12.6f} {row["R2_improvement"]:>8.6f} {p_str:>12}")

    print()
    print("\U0001f4ca INTERPRETATION:")
    print("  - If \u0394R\u00b2 > 0.01: physical load adds meaningful explanatory power")
    print("  - If R\u00b2_physical_only \u2248 R\u00b2_pressure: physical load confounds the effect")
    print("  - If p(physical) < 0.05: physical load is a significant independent predictor")
    print()

else:
    print("\u26a0\ufe0f  Physical load data not available \u2014 run pipeline + merge_outputs first.")


In [ ]:
# --- Physical load correlations: fatigue measures \u00d7 cognitive signals \u00d7 physical metrics ---
if existing_physical:
    target_signals_all = target_signals + existing_physical
    all_measures = measure_cols + existing_physical

    corr = df_with_loads[all_measures].corr()

    fig, ax = plt.subplots(figsize=(11, 9))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".3f", cmap="RdBu_r",
                vmin=-1, vmax=1, center=0, square=True,
                linewidths=0.5, ax=ax)

    tick_labels = all_measures
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(tick_labels, rotation=0, fontsize=8)
    ax.set_title("Correlations: Fatigue \u00d7 Cognitive Signals \u00d7 Physical Load", fontweight="bold", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "physical_load_correlation_heatmap.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved: physical_load_correlation_heatmap.png")

    print("\n" + "=" * 70)
    print("KEY CORRELATIONS: PHYSICAL LOAD \u2194 TARGET SIGNALS")
    print("=" * 70)
    for phys_col in existing_physical:
        print(f"\n  {phys_col}:")
        for sig in target_signals:
            r = corr.loc[phys_col, sig]
            strength = "strong" if abs(r) > 0.3 else ("moderate" if abs(r) > 0.15 else "weak")
            print(f"    vs {sig:<25s}  r = {r:+.4f}  ({strength})")
else:
    print("\u26a0\ufe0f  Physical load data not available \u2014 run pipeline + merge_outputs first.")


## 7. Key Findings: Fatigue Load Analysis

Summary of which fatigue load measure best predicts each defensive-quality signal.

In [ ]:
# --- Key Findings Summary ---
print("=" * 70)
print("FATIGUE LOAD ANALYSIS -- KEY FINDINGS")
print("=" * 70)
print()

print("1. BEST MEASURE PER SIGNAL")
print("-" * 50)
for _, row in best_per_signal.iterrows():
    improvement = ""
    if row["measure_code"] != "pressure_composite":
        baseline_r2 = results_df[(results_df["target_signal"] == row["target_signal"]) &
                                  (results_df["measure_code"] == "pressure_composite")]["r_squared"].values
        if len(baseline_r2) > 0 and not np.isnan(baseline_r2[0]) and not np.isnan(row["r_squared"]):
            if baseline_r2[0] > 0:
                pct = ((row["r_squared"] - baseline_r2[0]) / baseline_r2[0] * 100)
                improvement = f" (+{pct:.0f}% vs block load)"
    print(f"  {row['target_signal']:<30s} -> {row['measure']:<25s}  R2={row['r_squared']:.6f}{improvement}")

print()
print("2. MEASURE RANKING (mean R2 across all signals)")
print("-" * 50)
for rank, (m, r2) in enumerate(avg_r2.items(), 1):
    print(f"  #{rank}: {m:<30s}  mean R2 = {r2:.6f}")

print()
print("3. INTERPRETATION")
print("-" * 50)
dominant = avg_r2.index[0]
print(f"  The {dominant.lower()} measure has the highest average predictive power.")

# Per-signal narrative
for _, row in best_per_signal.iterrows():
    signal = row["target_signal"]
    measure = row["measure"]
    r2 = row["r_squared"]

    if "cumulative" in measure.lower():
        nature = "accumulated fatigue burden"
    elif "rolling" in measure.lower():
        nature = "recent sustained load"
    elif "decaying" in measure.lower():
        nature = "recent weighted load with memory"
    else:
        nature = "immediate block-level pressure"

    print(f"  {signal} is best predicted by {measure.lower()} (R2={r2:.5f}), "
          f"suggesting this signal is most sensitive to {nature}.")

print()
print("4. WHAT THIS REVEALS ABOUT FATIGUE DYNAMICS")
print("-" * 50)
interp = ("If cumulative measures (half/match) dominate:\n"
          "  Fatigue is ACCUMULATIVE: signals degrade as total load builds up.\n"
          "  Suggests substitution timing should be based on total load thresholds.\n\n"
          "If rolling/decaying measures dominate:\n"
          "  Fatigue is DRIVEN BY RECENT INTENSITY: the immediate past matters most.\n"
          "  Suggests high-intensity bursts cause more signal degradation.\n\n"
          "If block load (raw) dominates:\n"
          "  Fatigue is INSTANTANEOUS: only the current block's pressure matters.\n"
          "  Suggests pressure events cause immediate, temporary signal changes.\n")
print(interp)

# Save summary
findings = {
    "best_measure_per_signal": best_per_signal[["target_signal", "measure", "measure_code", "r_squared"]].to_dict(orient="records"),
    "measure_ranking": [{"measure": m, "avg_r2": float(r2)} for m, r2 in avg_r2.items()],
    "interpretation": f"The {dominant} measure has the highest average predictive power across all defensive-quality signals.",
}
with open(OUT_DIR / "fatigue_load_findings.json", "w") as f:
    json.dump(findings, f, indent=2, default=str)
print(f"Findings saved to: {OUT_DIR / 'fatigue_load_findings.json'}")

In [ ]:
# --- List all new output files ---
print("\nNew fatigue load analysis outputs:")
new_outputs = [
    "fatigue_load_predictive_power.csv",
    "fatigue_load_r2_comparison.png",
    "fatigue_load_best_measure_per_signal.png",
    "fatigue_load_trajectories.png",
    "fatigue_load_correlation_heatmap.png",
    "fatigue_load_findings.json",
]
for fname in new_outputs:
    fpath = OUT_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        print(f"  OK {fname:45s}  {size:>8,} bytes")
    else:
        print(f"  MISSING {fname:45s}")

In [ ]:
print("\nFatigue load analysis complete. All outputs saved to outputs/analysis/")